# Lab 08 · Human-in-the-loop and agent memory

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q --upgrade "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Review an answer and remember feedback

Python supplies a reviewer comment about a proposed outdoor activity. The notebook saves that comment in a list and passes it to the model with the earlier proposal. Read the revision and check whether it follows the saved feedback.

In [ ]:
proposal = "Go for a walk outside."
feedback = "It is raining; suggest an indoor activity."
print("Proposal:", proposal)
print("Simulated review: reject —", feedback)
notes = [feedback]
revision = ask(f"Earlier suggestion: {proposal}\nReview feedback: {notes[0]}\nSuggest one revision.")
print("Revised:", revision)

**Next:** Next, store approved advice with conditions that limit when it can be reused.

## Human feedback and memory: when is a retry safe?
A note that helped with one ShopEasy timeout could be dangerous in another case. Repeating a read-only price lookup has the same intended effect as doing it once. Repeating a card charge could charge the customer twice, even when the first request timed out.

In **human-in-the-loop** design, a person can review or correct a proposed decision. Here Python simulates that review so you can see what feedback does to the next model request. It approves retrying the lookup only when doing it again is safe and the fault is temporary.

The model receives **memory** as text that Python saves and adds to later prompts. Working notes hold feedback during a review. The archive holds approved advice for future cases in this notebook runtime. You will see a similar-looking payment timeout and check which advice it is allowed to receive.

**At a glance**

![Lab 08 at a glance: Council proposal (retry or do not retry), then Human review (verdict + reason), then Council revises (reads short-term notes), then Long-term memory (lesson + explicit scope), then Recall (similar AND applicable), then Held-out score (before vs after). Then back to Human review: at most 2 reviews.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/08.svg)

## Think about the example

Pricing and Catalog perform read-only lookups, while Payments performs a card charge. A timed-out charge may already have succeeded, so repeating it blindly can charge twice.

**Concepts:** Idempotent reads versus non-idempotent charges; Bounded retries and deduplicated effects.

Use the scenario above to consider these questions before running the code:

1. Why must a timeout not automatically trigger another payment charge?
2. Why does semantic similarity alone not establish that a stored lesson applies?

## The idea in context

Human review can correct a proposal and supply a scoped lesson. This lab uses a scripted reviewer, and retrieval checks both relevance and applicability.

Similarity is not applicability. Check known scope conditions before inserting a lesson into a prompt.

| Key term | Meaning |
| --- | --- |
| **Human in the loop** | A design in which a person approves or corrects the system's decisions. |
| **Short-term memory** | Working notes for the current task, passed in the prompt. |
| **Long-term memory** | Lessons stored across tasks and recalled when a similar case appears. |
| **Held-out case** | A case not used for feedback, reserved to test whether the approach generalizes. |

## Pattern · Human in the Loop

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Agents can propose actions whose consequences are hard to reverse. |
| **Problem** | Fully autonomous execution treats routine and high-risk decisions alike, and leaves no accountable decision point. |
| **Solution** | Classify each step by risk, run the low-risk ones, and gate high-risk ones behind explicit human approval, with an audit trail. |
| **Use when** | Decisions affect people or money; reversibility varies; auditability is required. |
| **Trade-off** | Safer and accountable, but approvals interrupt the flow and can become a bottleneck. |
| **Flow** | Plan → Classify risk → Approve? → Execute + audit |
| **Quality attributes** | Benefits: Safety, Reliability, Auditability, Usability. Liabilities: Latency, Cost, Scalability, Usability, Reliability. |
| **In this lab** | Python simulates the human architect and returns approval or rejection with reasons. Self-study Lab 10 shows how to route cases that require judgment to a review queue. |

## Pattern · Scratchpad Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Multi-step work produces partial results and pending questions that are not long-term facts. |
| **Problem** | Keeping work in progress only in the message history makes it noisy, fragile, and hard to update cleanly. |
| **Solution** | Give the agent a dedicated block of working notes that it appends to, replaces, or clears as the work evolves. |
| **Use when** | Tasks span several turns; intermediate state matters; working notes should stay separate from durable facts. |
| **Trade-off** | Makes state explicit, but unmanaged notes consume context and can preserve obsolete assumptions. |
| **Flow** | Work → Scratchpad → Update notes → Resume |
| **Quality attributes** | Benefits: Reliability, Observability, Maintainability. Liabilities: Cost, Reliability, Privacy. |
| **In this lab** | `short_term` holds the simulated reviewer's feedback while one decision is made. The notebook includes it in the revision prompt. Once the decision is archived, the scratchpad is cleared, so an obsolete note cannot leak into the next case. |

## Pattern · Archival Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Long-lived agents accumulate more knowledge than can stay in the context window. |
| **Problem** | Keeping everything in context is expensive and eventually impossible, while dropping old facts loses useful knowledge. |
| **Solution** | Move durable facts into archival storage, and search it semantically so that only the relevant facts return to the context. |
| **Use when** | Facts accumulate over time; semantic retrieval is acceptable; long-term persistence matters. |
| **Trade-off** | Scales far beyond the context, but retrieval can miss relevant facts or return misleading neighbours. |
| **Flow** | Fact → Archive → Semantic search → Recall into context |
| **Quality attributes** | Benefits: Scalability, Cost, Data freshness. Liabilities: Latency, Reliability, Maintainability. |
| **In this lab** | The accepted decision and its reasons are archived in a Chroma collection and recalled by semantic search. Step 4 shows the pattern's liability: a similar timeout may hide a non-idempotent effect. |

## Main exercise at a glance

1. A proposal and a policy
2. Simulate human review and define its scope
3. Archive the lesson with an applicability condition
4. Hold out two new cases
5. Measure before and after, without requiring a win

## Step 1 · A proposal and a policy
ShopEasy's Pricing service reads a quote. The lookup is **idempotent**, so repeating it has the same intended effect as doing it once. A **transient** failure is temporary. These two facts are part of the classroom case.

The first proposal says `do not retry`. `POLICY` gives the rule the model will receive on later calls. Here `retry` means trying again with a limit, a delay between attempts, and some variation in those delays to avoid retrying all at once.

In [ ]:
case = "ShopEasy Pricing: an idempotent quote lookup timed out during a transient network fault."
proposal = {"option": "do not retry", "reason": "A timeout leaves an uncertain outcome."}
OPTIONS = ["retry", "do not retry"]
POLICY = "Retry only transient idempotent effects with approved advice; otherwise do not retry."
print("Proposed:", proposal)

## Step 2 · Simulate human review and define its scope
Python acts as the reviewer for this particular Pricing case. It rejects the initial proposal, adds the reason to the `short_term` list, and asks the model for a revised label using that note.

Read the printed verdicts. When the model answers `retry`, the next review accepts it and records two facts that limit reuse, `idempotent=True` and `transient=True`. The loop reviews the first proposal and allows up to two revisions. Every revision gets a verdict. If the output says there was no accepted proposal, inspect the replies and rerun Step 2 before moving on. Try a fresh run of Step 2 if the model answers with prose where the cell expects a label.

In [ ]:
option, short_term = proposal["option"], []
for n in range(1, 4):
    if option == "do not retry":
        review = {"verdict": "reject", "reason": "This read is idempotent and the fault is transient; use bounded backoff and jitter."}
    else:
        review = {"verdict": "accept", "reason": "A bounded retry is safe for this transient failed read.", "idempotent": True, "transient": True}
    print(f"Review {n}: {option} -> {review['verdict']}: {review['reason']}")
    if review["verdict"] == "accept" or n == 3:
        break
    short_term.append(review["reason"])
    notes = "Human feedback on this case: " + "\n".join(short_term)
    prompt = (f"Policy: {POLICY}\n{notes}\nCase: {case}\n"
              f"Reply with exactly one label from {OPTIONS}. Retry means bounded backoff with jitter.")
    option = ask(prompt).strip().rstrip(".").lower()
    if option not in OPTIONS:
        raise ValueError(f"Unknown option: {option!r}")
assert review["verdict"] == "accept", "No accepted proposal; the human must decide directly."

## Step 3 · Archive the lesson with an applicability condition
The notebook stores the case text, the approved advice, and the two approved conditions in Chroma. **Metadata** is the extra information stored alongside the searchable case. The case text is used to find similar situations. The advice and conditions are saved as metadata.

This is the same kind of search by meaning as Lab 02. It compares **embeddings**, lists of numbers that represent the text. `recall` returns any matching advice with its distance score, where a smaller number means a closer match. A candidate must also match both of the known facts about the new case to enter the prompt. The `candidates_only=True` option lets you inspect similarity without applying the lesson.

The first Chroma operation may take longer while it downloads the embedding model. Look for `Working notes: 0 | archived lessons: 1`. The notes were cleared after archiving. The archive stays available during the current runtime, and restarting the runtime clears it.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

db = chromadb.Client(Settings(anonymized_telemetry=False))
memory = db.get_or_create_collection("retry_lessons", embedding_function=DefaultEmbeddingFunction(),
                                    configuration={"hnsw": {"space": "cosine"}})
lesson = "\n".join([f"Past case: {case}", *short_term, f"Accepted: {option}. {review['reason']}"])
memory.upsert(ids=["D1"], documents=[case], metadatas=[{"lesson": lesson, "idempotent": review["idempotent"], "transient": review["transient"]}])
short_term.clear()
print("Working notes:", len(short_term), "| archived lessons:", memory.count())

def recall(new_case, idempotent=None, transient=None, max_distance=0.6, *, candidates_only=False):
    if not candidates_only and (type(idempotent) is not bool or type(transient) is not bool):
        return []  # unknown scope cannot receive advice
    where = None
    if not candidates_only:
        where = {"$and": [{"idempotent": idempotent}, {"transient": transient}]}
    found = memory.query(query_texts=[new_case], n_results=1, where=where)
    results = []
    for distance, record in zip(found["distances"][0], found["metadatas"][0]):
        if distance <= max_distance:
            results.append((round(distance, 2), record["lesson"]))
    return results

print(recall(case, idempotent=True, transient=True)[0][1])

## Step 4 · Hold out two new cases
A **held-out case** is saved for checking the result. Neither of these cases supplied the feedback in Step 2. Each tuple has the new case, two facts from its service contract, and an expected label for scoring later.

Read the `Semantic candidates` and `applicable` counts beside each case. Catalog is another idempotent read, so it can receive the lesson if the texts are close enough. Payments might look similar in a search, but the charge is non-idempotent and must get zero applicable lessons. Python checks the out-of-scope cases before running any new model calls.

In [ ]:
held_out = [
    ("ShopEasy Catalog: an idempotent product lookup timed out during a transient network fault.", True, True, "retry"),
    ("ShopEasy Payments: a non-idempotent card charge timed out; the charge outcome is uncertain.", False, True, "do not retry"),
]
for c, safe, transient, _ in held_out:
    print("Semantic candidates:", len(recall(c, candidates_only=True)), "| applicable:", len(recall(c, idempotent=safe, transient=transient)), "|", c)
assert not recall(held_out[1][0], idempotent=False, transient=True), "A read-only retry lesson must not reach the payment case."
assert not recall(held_out[0][0], idempotent=None, transient=True), "Unknown scope needs review."
assert not recall(held_out[0][0], idempotent=True, transient=False), "Permanent failures need different advice."

## Step 5 · Measure before and after, without requiring a win
The `before` prompt contains the policy and case. The policy calls for approved advice before a retry, so the model can reasonably choose `do not retry` for Catalog until it sees the approved lesson. The expected labels in Step 4 reflect the classroom reviewer's judgment with that approval in mind.

If a case receives applicable advice, ask again with that advice added and show the two replies, the expected label, and whether each matched. If a case receives no advice, reuse its first answer because the prompt would be identical. Payments acts as the unchanged case. Score changes apply to this small example. Repeating the experiment can give different model replies.

In [ ]:
def recommend(case_text, notes=""):
    prompt = (f"Policy: {POLICY}\n{notes}Case: {case_text}\n"
              f"Reply with exactly one label from {OPTIONS}. Retry means bounded backoff with jitter.")
    answer = ask(prompt).strip().rstrip(".").lower()
    if answer not in OPTIONS:
        raise ValueError(f"Unknown option: {answer!r}")
    return answer

scores = {"before": 0, "after": 0}
for c, safe, transient, expected in held_out:
    assert case != c, "Feedback cases cannot be test cases."
    before = recommend(c)
    lessons = [text for _, text in recall(c, idempotent=safe, transient=transient)]
    if lessons:
        notes = "Applicable approved lessons:\n" + "\n".join(lessons) + "\n"
        after = recommend(c, notes)
        comparison = "new reply with approved advice"
    else:
        after = before
        comparison = "reused first reply because the prompt is unchanged"
    print(c)
    print(f"  Expected: {expected} | before: {before} | after: {after} ({comparison})")
    before_match, after_match = before == expected, after == expected
    print(f"  Matched expected label? Before: {before_match} | after: {after_match}")
    scores["before"] += before_match
    scores["after"] += after_match
for run, score in scores.items():
    print(f"{run}: {score}/{len(held_out)} match the human")
print("Score change:", scores["after"] - scores["before"])

## What you should see

Step 2 prints the scripted review. Step 3 prints one archived lesson and an empty working-notes list. Step 4 checks that Payments and cases with unknown idempotency or permanent failures get no applicable advice. Step 5 prints the before and after labels for each case, notes whether it reused the first reply, and reports the change in score.

## Try it

1. Call `recall` on the Payments case with `candidates_only=True` to inspect similarity. Compare that result with the normal lookup using `idempotent=False` and `transient=True`.
2. Try `recall(case, idempotent=None, transient=True)`. What would you ask the service owner to learn whether repeating the operation is safe?
3. Test a stricter `max_distance` in a separate Catalog lookup. How does it affect which lesson is available?
4. Suppose a service may receive the same event twice. What information would help it recognize the duplicate before repeating an action?

## Takeaway

Working notes help the current review. Archived lessons help later cases when both the text match and the approved conditions fit. Check those conditions before inserting past advice into a new prompt.